# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# explorar datasets
orders.head(5)

In [ ]:
orders.info()
orders.describe()

In [ ]:
catalog.head(7)

In [ ]:
catalog.info()
catalog.describe()

In [ ]:
marketing.head(5)

In [ ]:
marketing.info()
marketing.describe()

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
orders_clean = orders.copy()
catalog_clean = catalog.copy()
marketing_clean = marketing.copy()

In [ ]:
# ===========================================================
# 1. orders_clean: Limpieza completa
# ===========================================================

print("=" * 60)
print("LIMPIEZA: orders")
print("=" * 60)

# ── 1.1 Convertir fecha_hora_pedido a datetime ──────────────
orders_clean["fecha_hora_pedido"] = pd.to_datetime(
    orders_clean["fecha_hora_pedido"], errors="coerce"
)
print(f"[OK] fecha_hora_pedido → datetime. "
      f"Nulos generados: {orders_clean['fecha_hora_pedido'].isna().sum()}")

In [ ]:
# ── 1.2 Convertir cantidad a entero ─────────────────────────
# Primero verificamos valores antes de castear
print(f"\nValores únicos no numéricos en 'cantidad' (muestra):")
non_numeric_qty = orders_clean[~orders_clean["cantidad"].apply(
    lambda x: str(x).replace(".", "", 1).replace("-", "", 1).isdigit()
)]["cantidad"].unique()
print(non_numeric_qty[:10])

In [ ]:
orders_clean["cantidad"] = pd.to_numeric(orders_clean["cantidad"], errors="coerce")
print(f"NaN tras conversión numérica: {orders_clean['cantidad'].isna().sum()}")

In [ ]:
orders_clean["cantidad"].describe()

In [ ]:
# ── 1.3 Eliminar filas con cantidad <= 0 (inválidas) ────────

mask_invalida = orders_clean["cantidad"].isna() | (orders_clean["cantidad"] <= 0)
orders_clean = orders_clean[~mask_invalida].copy()

In [ ]:
# Winsorizarr ANTES de convertir a int (sobre float64)
from scipy.stats import mstats

print(f"Antes winsorización — max: {orders_clean['cantidad'].max()}")
orders_clean["cantidad"] = mstats.winsorize(
    orders_clean["cantidad"].astype(float),  # forzar float64
    limits=[0.01, 0.01]
).data
print(f"Después winsorización — max: {orders_clean['cantidad'].max()}")
print(orders_clean["cantidad"].describe())

In [ ]:
# Convertir a int DESPUÉS de winsorizarr
orders_clean["cantidad"] = orders_clean["cantidad"].astype(int)
print(f"[OK] cantidad → int. Dtype: {orders_clean['cantidad'].dtype}")

In [ ]:
# Diagnóstico previo al filtro
print(f"Shape actual: {orders_clean.shape}")
print(f"NaN en cantidad: {orders_clean['cantidad'].isna().sum()}")
print(f"Valores <= 0: {(orders_clean['cantidad'] <= 0).sum()}")
print(f"Dtype cantidad: {orders_clean['cantidad'].dtype}")

In [ ]:
# ── 1.4 Verificar consistencia de montos ────────────────────
# monto_total debería ser aprox: cantidad * precio_unitario - monto_descuento
orders_clean["monto_calculado"] = (
    orders_clean["cantidad"] * orders_clean["precio_unitario"] - orders_clean["monto_descuento"]
)
tolerancia = 0.01  # tolerancia de redondeo
inconsistencias = (
    (orders_clean["monto_total"] - orders_clean["monto_calculado"]).abs() > tolerancia
).sum()
print(f"\nInconsistencias en monto_total (vs. cálculo): {inconsistencias}")

if inconsistencias > 0:
    print("  → Corrigiendo monto_total con el valor calculado...")
    orders_clean.loc[
        (orders_clean["monto_total"] - orders_clean["monto_calculado"]).abs() > tolerancia,
        "monto_total"
    ] = orders_clean["monto_calculado"]

orders_clean.drop(columns=["monto_calculado"], inplace=True)

In [ ]:
# Eliminar filas con precio_unitario o monto_total negativos o nulos
mask_montos_invalidos = (
    orders_clean["precio_unitario"].le(0) |
    orders_clean["monto_total"].isna()
)
print(f"Filas con montos inválidos: {mask_montos_invalidos.sum()}")
orders_clean = orders_clean[~mask_montos_invalidos].copy()

In [ ]:
# ── 1.5 Eliminar duplicados ─────────────────────────────────
duplicados = orders_clean.duplicated().sum()
print(f"\nDuplicados exactos encontrados: {duplicados}")
orders_clean.drop_duplicates(inplace=True)

# Duplicados por id_pedido (pedidos con el mismo ID)
dup_id = orders_clean.duplicated(subset=["id_pedido"]).sum()
print(f"id_pedido duplicados: {dup_id}")
if dup_id > 0:
    orders_clean.drop_duplicates(subset=["id_pedido"], keep="first", inplace=True)
    print("  → Conservada primera ocurrencia por id_pedido")

In [ ]:
# ── 1.6 Estandarizar variables categóricas ──────────────────
categoricas = ["pais", "dispositivo", "fuente_referencia",
               "nombre_producto", "categoria_producto"]

for col in categoricas:
    if col in orders_clean.columns:
        antes = orders_clean[col].nunique()
        orders_clean[col] = (
            orders_clean[col]
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
        )
        despues = orders_clean[col].nunique()
        print(f"  [{col}] categorías antes: {antes} → después: {despues}")

print(f"\n[ORDERS] Shape final: {orders_clean.shape}")
print(f"Nulos restantes:\n{orders_clean.isnull().sum()}\n")

In [ ]:
# Eliminar filas con nulos restantes
filas_antes = len(orders_clean)
orders_clean = orders_clean.dropna().copy()
filas_eliminadas = filas_antes - len(orders_clean)
print(f"Filas eliminadas por nulos restantes: {filas_eliminadas}")
print(f"Shape final: {orders_clean.shape}")
print(f"Nulos restantes:\n{orders_clean.isnull().sum()}")

In [ ]:
orders_clean.info()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Antes vs Después — limpieza orders_clean", fontsize=14)

cols = ["cantidad", "precio_unitario", "monto_descuento", "monto_total"]

for i, col in enumerate(cols):
    # Antes — df crudo
    axes[0, i].boxplot(orders[col].dropna(), patch_artist=True,
                       boxprops=dict(facecolor="#F0997B"))
    axes[0, i].set_title(f"{col}\nANTES", fontsize=10)

    # Después — df limpio
    axes[1, i].boxplot(orders_clean[col].dropna(), patch_artist=True,
                       boxprops=dict(facecolor="#5DCAA5"))
    axes[1, i].set_title(f"{col}\nDESPUÉS", fontsize=10)

plt.tight_layout()
plt.show()

**📋 Hallazgos y tratamiento — orders_clean**

**Transformaciones aplicadas**
- `fecha_hora_pedido` convertida de object → datetime64
- `cantidad` convertida de object → float64 → int64

**Problemas detectados y corregidos**
| Problema | Filas afectadas | Acción |
|---|---|---|
| Cantidad NaN o ≤ 0 | 54 | Eliminadas |
| Duplicados exactos | 100 | Eliminados |
| Inconsistencias en monto_total | 1.156 | Corregidas con cantidad × precio − descuento |
| Nulos en variables categóricas | 346 | Eliminadas |

**Tratamiento de outliers**
Se identificaron valores extremos en columnas numéricas mediante
boxplot. Se aplicó winsorización al 1% y 99% sobre float64
**antes** de convertir a int para evitar comportamiento inesperado.

| Columna | Antes (max) | Después (max) | Impacto |
|---|---|---|---|
| `cantidad` | 20.000 | 2 | Errores de captura eliminados |
| `monto_total` | 9.000.000usd | 1.000usd | Valores extremos corregidos |
| `precio_unitario` | outliers altos | distribución compacta | Mejora notable |
| `monto_descuento` | outliers leves | sin cambios dramáticos | Era la más limpia |

**Estandarización categóricas**
- `pais`: 6 valores → 3 categorías reales (espacios y mayúsculas)
- `dispositivo`, `fuente_referencia`, `nombre_producto`,
  `categoria_producto` estandarizadas a lowercase sin espacios extras

**Resultado final**
- **Shape original:** 25.100 filas × 12 columnas
- **Shape final:** 24.600 filas × 12 columnas
- **Registros depurados:** 500 (2% del total)
- **Calidad final:** 0 nulos · 0 duplicados · tipos correctos ✓
- Outliers tratados con winsorización al 1%/99% ✓
- **Calidad final:** 0 nulos · 0 duplicados · tipos correctos ✓

In [ ]:
# ===========================================================
# 2. catalog_clean — Validación (datos limpios)
# ===========================================================
print("=" * 60)
print("VALIDACIÓN: catalog (se espera datos limpios)")
print("=" * 60)

print(f"Shape: {catalog_clean.shape}")
print(f"Nulos:\n{catalog_clean.isnull().sum()}")
print(f"Duplicados: {catalog_clean.duplicated().sum()}")
print(f"Resumen costo_unitario:\n{catalog_clean['costo_unitario'].describe()}")

In [ ]:
# Verificar precios positivos
precios_invalidos = (catalog["costo_unitario"] <= 0).sum()
print(f"Precios <= 0: {precios_invalidos}")
print("[OK] catalog validado — sin intervención requerida.\n")

In [ ]:
# Estandarizar todas las categóricas en catalog_clean
categoricas_catalog = ["nombre_producto", "categoria_producto", "proveedor"]

for col in categoricas_catalog:
    catalog_clean[col] = (
        catalog_clean[col]
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    print(f"[OK] {col}: {catalog_clean[col].unique()}")

**📋 Hallazgos y tratamiento — catalog_clean**

**Transformaciones aplicadas**
- `nombre_producto`, `categoria_producto` y `proveedor`
  estandarizados a lowercase, sin espacios extras

**Decisión**
Aunque catalog no presentó nulos ni duplicados, se aplicó
estandarización de categóricas para garantizar integridad
referencial en los merges con orders_clean en etapas posteriores

**Resultado final**
- **Shape:** 7 filas × 4 columnas
- **Calidad final:** 0 nulos · 0 duplicados · tipos correctos ✓
- Categóricas estandarizadas para compatibilidad entre datasets ✓

In [ ]:
# ===========================================================
# 3: marketing_clean — Limpieza
# ===========================================================
print("=" * 60)
print("LIMPIEZA: marketing")
print("=" * 60)

# ── 3.1 Convertir fecha a datetime
marketing_clean["fecha"] = pd.to_datetime(
    marketing_clean["fecha"], errors="coerce"
)
print(f"[OK] fecha → datetime. "
      f"Nulos generados: {marketing_clean['fecha'].isna().sum()}")
# Resultado: 0 → no se requiere eliminación

In [ ]:
# ── 3.2 Tratar nulos en columna 'canal'
nulos_canal = marketing_clean["canal"].isna().sum()
total = len(marketing_clean)
pct_nulos = nulos_canal / total * 100
print(f"Nulos en 'canal': {nulos_canal} ({pct_nulos:.1f}%)")
print(marketing_clean["canal"].value_counts(dropna=False))

In [ ]:
# DECISIÓN: aunque pct > 5%, id_campaña permite inferir el canal
# Se opta por inferencia para preservar los 101 registros
print("\n→ id_campaña permite inferir canal. Aplicando inferencia...")

In [ ]:
def inferir_canal(row):
    if pd.notna(row["canal"]):
        return row["canal"]
    idc = str(row.get("id_campaña", "")).lower()
    if "organic"     in idc: return "organic"
    if "paid_search" in idc: return "paid_search"
    if "social"      in idc: return "social"
    return np.nan

marketing_clean["canal"] = marketing_clean.apply(inferir_canal, axis=1)
print(f"Nulos tras inferencia: {marketing_clean['canal'].isna().sum()}")
print(marketing_clean["canal"].value_counts())

In [ ]:
# ── 3.3 Estandarizar todas las categóricas en marketing_clean ──────────────────────────────────
categoricas_marketing = ["pais", "id_campaña", "canal"]

for col in categoricas_marketing:
    marketing_clean[col] = (
        marketing_clean[col]
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    print(f"[OK] {col}: {marketing_clean[col].unique()}")

In [ ]:
# ── 3.4 Verificar calidad de gasto ──────────────────────────
gasto_negativo = (marketing_clean["gasto"] < 0).sum()
gasto_nulo     = marketing_clean["gasto"].isna().sum()
print(f"\nGasto negativo: {gasto_negativo} | Gasto nulo: {gasto_nulo}")
if gasto_negativo > 0:
    marketing_clean = marketing_clean[marketing_clean["gasto"] >= 0].copy()
    print("  → Filas con gasto negativo eliminadas.")

In [ ]:
# ── 3.5 Duplicados ──────────────────────────────────────────
dup_mkt = marketing_clean.duplicated().sum()
print(f"Duplicados: {dup_mkt}")
marketing_clean.drop_duplicates(inplace=True)

print(f"\n[MARKETING] Shape final: {marketing_clean.shape}")
print(f"Nulos restantes:\n{marketing_clean.isnull().sum()}\n")

In [ ]:
marketing_clean.info()

**📋 Hallazgos y tratamiento — marketing_clean**

**Transformaciones aplicadas**
- `fecha` convertida de object → datetime64
- `canal` inferido desde `id_campaña` para registros sin valor
- `pais`, `id_campaña` y `canal` estandarizados a lowercase

**Problemas detectados y corregidos**
| Problema | Filas afectadas | Acción |
|---|---|---|
| Nulos en `canal` | 101 (6.2%) | Inferidos desde `id_campaña` |
| Gasto negativo | 0 | Sin intervención |
| Duplicados | 0 | Sin intervención |

**Decisión sobre nulos en `canal`**
Aunque el porcentaje (6.2%) superaba el umbral conservador del 5%,
se optó por **inferencia** en lugar de eliminación porque `id_campaña`
contenía el canal explícitamente en su nombre:
- `social_Argentina` → social
- `organic_Mexico` → organic  
- `paid_search_Colombia` → paid_search

**Distribución final de `canal`**
| Canal | Antes | Después |
|---|---|---|
| paid_search | 507 | 540 |
| organic | 506 | 540 |
| social | 506 | 540 |
| NaN | 101 | 0 |

> Los 101 nulos se distribuyeron uniformemente entre los 3 canales,
> confirmando que la ausencia era aleatoria y sin sesgo.

**Resultado final**
- **Shape original:** 1.620 filas × 5 columnas
- **Shape final:** 1.620 filas × 5 columnas
- **Registros recuperados:** 101 (0% de pérdida)
- **Calidad final:** 0 nulos · 0 duplicados · tipos correctos ✓
- Categóricas estandarizadas para compatibilidad entre datasets ✓

In [ ]:
# ===========================================================
# RESUMEN EJECUTIVO
# ===========================================================
print("=" * 60)
print("RESUMEN EJECUTIVO — Calidad de datos post-limpieza")
print("=" * 60)

resumen = pd.DataFrame({
    "DataFrame": ["orders", "catalog", "marketing"],
    "Filas_finales": [len(orders_clean), len(catalog_clean), len(marketing_clean)],
    "Nulos_totales": [
        orders_clean.isnull().sum().sum(),
        catalog_clean.isnull().sum().sum(),
        marketing_clean.isnull().sum().sum(),
    ],
    "Duplicados": [0, 0, 0],  # ya eliminados
})
print(resumen.to_string(index=False))
print()
print(f"Tras la revisión, depuración y estandarización de orders, catalog y marketing, los datos quedaron validados y listos para análisis.")
print(f"Con ello, se cumple la solicitud inicial del negocio: trabajar sobre bases confiables para evaluar revenue, costos, rentabilidad y desempeño posterior.")

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# Exportar datasets limpios
orders_clean.to_csv('orders_clean.csv', index=False, decimal=',')
catalog_clean.to_csv('catalog_clean.csv', index=False, decimal=',')
marketing_clean.to_csv('marketing_clean.csv', index=False, decimal=',')

print(f"orders_clean:    {len(orders_clean)} filas")
print(f"catalog_clean:   {len(catalog_clean)} filas")
print(f"marketing_clean: {len(marketing_clean)} filas")

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# ============================================================
# PARTE 1 — Rentabilidad
# ============================================================
print("=" * 60)
print("PARTE 1: Rentabilidad del negocio")
print("=" * 60)

# ── KPI 1: Ingreso total (revenue) ──────────────────────────
revenue_total = orders_clean["monto_total"].sum()
print(f"\nKPI 1 — Ingreso total (revenue): ${revenue_total:,.2f}")


In [ ]:
# ── KPI 2: Costo total ──────────────────────────────────────
# Cruzar orders con catalog para obtener costo_unitario por producto
orders_catalog = orders_clean.merge(
    catalog_clean[["nombre_producto", "costo_unitario"]],
    on="nombre_producto",
    how="left"
)
orders_catalog["costo_total_linea"] = (
    orders_catalog["cantidad"] * orders_catalog["costo_unitario"]
)
costo_total = orders_catalog["costo_total_linea"].sum()
print(f"KPI 2 — Costo total:             ${costo_total:,.2f}")

In [ ]:
# ── KPI 3: Inversión en marketing ───────────────────────────
inversion_marketing = marketing_clean["gasto"].sum()
print(f"KPI 3 — Inversión en marketing:  ${inversion_marketing:,.2f}")

In [ ]:
# ── KPI 4: Profit y margen ──────────────────────────────────
profit = revenue_total - costo_total - inversion_marketing
margen = (profit / revenue_total) * 100
print(f"\nKPI 4 — Profit neto:             ${profit:,.2f}")
print(f"         Margen de rentabilidad:  {margen:.1f}%")
print(f"         ¿Es rentable?:           {'✓ SÍ' if profit > 0 else '✗ NO'}")

In [ ]:
# ============================================================
# PARTE 2 — Comportamiento de ventas
# ============================================================
print("\n" + "=" * 60)
print("PARTE 2: Comportamiento de ventas")
print("=" * 60)

# ── KPI 5: Ticket promedio por orden ────────────────────────
ticket_promedio = orders_clean.groupby("id_pedido")["monto_total"].sum().mean()
print(f"\nKPI 5 — Ticket promedio por orden:          ${ticket_promedio:,.2f}")

In [ ]:
# ── KPI 6: Cantidad promedio de productos por orden ─────────
cantidad_promedio = orders_clean.groupby("id_pedido")["cantidad"].sum().mean()
print(f"KPI 6 — Cantidad promedio por orden:         {cantidad_promedio:.1f} productos")

In [ ]:
# ── KPI 7: Producto más vendido ─────────────────────────────
producto_mas_vendido = (
    orders_clean.groupby("nombre_producto")["cantidad"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
print(f"\nKPI 7 — Top 5 productos más vendidos:")
print(producto_mas_vendido.to_string())

---

In [ ]:
# ── KPI 8: Gasto en marketing por canal ─────────────────────
gasto_por_canal = (
    marketing_clean.groupby("canal")["gasto"]
    .sum()
    .sort_values(ascending=False)
)
print(f"\nKPI 8 — Gasto en marketing por canal:")
print(gasto_por_canal.to_string())

**📊 KPIs principales — Resultados**

**Parte 1: Rentabilidad del negocio**

| KPI | Valor | Detalle |
|---|---|---|
| Revenue total | 9,497,385.83usd | Ingreso bruto por ventas |
| Costo total | 3,789,092.40usd | Costo de productos vendidos |
| Inversión marketing | 2,871,843.53usd | Gasto total en campañas |
| **Profit neto** | **2,836,449.90usd** | Revenue − costo − marketing |
| **Margen de rentabilidad** | **29.9%** | Por cada 100usd vendidos, 29.9usd es ganancia |
| **¿Es rentable?** | **✓ SÍ** | Profit positivo confirmado |


**Estructura del revenue**
| Concepto | Monto | % del Revenue |
|---|---|---|
| Costo de producto | 3,789,092usd | 39.9% |
| Inversión marketing | 2,871,843usd | 30.2% |
| **Profit neto** | **2,836,449usd** | **29.9%** |

**Parte 2: Comportamiento de ventas**


| KPI | Valor | Lectura |
|---|---|---|
| Ticket promedio por orden | 386.07usd | Valor razonable para ecommerce |
| Cantidad promedio por orden | 1.5 productos | Comportamiento retail normal |
| Producto más vendido | vacuum-pro-black (6.203 u.) | Líder real tras limpiar outliers |

**Top 5 productos más vendidos**
| Producto | Unidades |
|---|---|
| vacuum-pro-black | 6.203 |
| jacket-winter-m | 6.185 |
| blender-xl-red | 6.184 |
| sneakers-urban-42 | 6.085 |
| laptop-gaming-16gb | 4.180 |

**Gasto en marketing por canal**
| Canal | Gasto | % del total |
|---|---|---|
| social | 976,818.37usd | 34.0% |
| organic | 972,650.96usd | 33.9% |
| paid_search | 922,374.20usd | 32.1% |

> La inversión por canal es casi uniforme, sin apuesta clara
> por un canal específico. Esto puede ser una oportunidad
> de optimización según rendimiento por canal.

**RappiPlus es rentable**. Los KPIs muestran profit neto positivo y un margen de 29.9%, lo que confirma que el negocio genera valor después de cubrir costo de producto e inversión en marketing.

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:

# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()


In [ ]:
# 1. Conocer el tamaño de la tabla
print(f"Shape: {events.shape}")
events.info()

In [ ]:
# 2. Duplicados exactos
dup_exactos = events.duplicated().sum()
print(f"\nDuplicados exactos: {dup_exactos}")

In [ ]:
# 3. Duplicados por usuario + sesión + evento
dup_sesion = events.duplicated(
    subset=["id_usuario", "id_sesion", "nombre_evento"]
).sum()
print(f"Duplicados por usuario+sesión+evento: {dup_sesion}")

In [ ]:
# 4. Verificar nulos
print(events.isnull().sum())

In [ ]:
# Eventos únicos y su conteo
print(f"\nEventos únicos en nombre_evento:")
print(events["nombre_evento"].value_counts())
print(f"\nTotal eventos: {events['nombre_evento'].value_counts().sum()}")
print(f"Coincide con shape: {events['nombre_evento'].value_counts().sum() == len(events)}")

In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit'      THEN 1
        WHEN 'add_to_cart'      THEN 2
        WHEN 'select_item'      THEN 3
        WHEN 'begin_checkout'   THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase'         THEN 6
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos,
    ROUND(
        COUNT(DISTINCT id_usuario) * 100.0 /
        FIRST_VALUE(COUNT(DISTINCT id_usuario))
        OVER (ORDER BY
            CASE nombre_evento
                WHEN 'first_visit'      THEN 1
                WHEN 'add_to_cart'      THEN 2
                WHEN 'select_item'      THEN 3
                WHEN 'begin_checkout'   THEN 4
                WHEN 'add_payment_info' THEN 5
                WHEN 'purchase'         THEN 6
            END), 2
    ) AS conversion_vs_inicio,
    ROUND(
        COUNT(DISTINCT id_usuario) * 100.0 /
        LAG(COUNT(DISTINCT id_usuario))
        OVER (ORDER BY
            CASE nombre_evento
                WHEN 'first_visit'      THEN 1
                WHEN 'add_to_cart'      THEN 2
                WHEN 'select_item'      THEN 3
                WHEN 'begin_checkout'   THEN 4
                WHEN 'add_payment_info' THEN 5
                WHEN 'purchase'         THEN 6
            END), 2
    ) AS conversion_vs_etapa_anterior
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit'      THEN 1
        WHEN 'add_to_cart'      THEN 2
        WHEN 'select_item'      THEN 3
        WHEN 'begin_checkout'   THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase'         THEN 6
    END;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

etapas = conversion["nombre_evento"].tolist()
usuarios = conversion["usuarios_unicos"].tolist()
colores = ["#7F77DD", "#5DCAA5", "#5DCAA5", "#5DCAA5", "#D85A30", "#5DCAA5"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(etapas[::-1], usuarios[::-1], color=colores[::-1])

# Etiquetas
for bar, u, c in zip(bars, usuarios[::-1], conversion["conversion_vs_inicio"].tolist()[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f"{u:,} usuarios ({c}%)", va="center", fontsize=8)

ax.set_xlabel("Usuarios únicos")
ax.set_title("Funnel de conversión — RappiPlus", fontsize=10)
ax.set_xlim(0, 9500)
plt.tight_layout()
plt.show()

**📋 Hallazgos — Funnel de conversión**

**Hallazgos principales**

El análisis del funnel de conversión permitió identificar que la principal pérdida de usuarios ocurre entre`begin_checkout` y `add_payment_info`, con una conversión de 86.71% frente a la etapa anterior.
En cambio, la transición de `add_payment_info` a `purchase` alcanza 99.84%, lo que indica que el mayor punto de fricción no está en la decisión final de compra, sino en el proceso de ingreso de datos de pago.
Por ello, la principal oportunidad de mejora se concentra en simplificar la experiencia de checkout antes de la compra final.

**💡 Recomendación**
Optimizar el formulario de pago: simplificar campos, agregar
métodos de pago alternativos y habilitar guardado de tarjetas
para usuarios recurrentes.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`


In [ ]:
# Explorar tabla users
# =========================
query_users = '''

SELECT *
FROM users;

'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;

'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Diagnóstico completo de ambas tablas
print("=== users ===")
users.info()

print("\n=== user_activity ===")
user_activity.info()

In [ ]:
# ¿Hasta cuántos días llega user_activity?
print(f"Máximo dias_despues_registro: {user_activity['dias_despues_registro'].max()}")
print(f"Mínimo: {user_activity['dias_despues_registro'].min()}")
print(user_activity['dias_despues_registro'].describe())

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH actividad AS (
    SELECT
        u.id_usuario,
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS mes_cohorte,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 1  AND 7
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w1,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 8  AND 14
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w2,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 15 AND 21
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w3,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 22 AND 28
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w4
    FROM users u
    LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
    GROUP BY u.id_usuario, mes_cohorte
),
resumen AS (
    SELECT
        mes_cohorte,
        COUNT(id_usuario)  AS total_usuarios,
        SUM(retenido_w1)   AS retenido_w1,
        SUM(retenido_w2)   AS retenido_w2,
        SUM(retenido_w3)   AS retenido_w3,
        SUM(retenido_w4)   AS retenido_w4
    FROM actividad
    GROUP BY mes_cohorte
)
SELECT
    mes_cohorte,
    total_usuarios,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    retenido_w4,
    ROUND(retenido_w1 * 100.0 / total_usuarios, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / total_usuarios, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / total_usuarios, 2) AS semana_3,
    ROUND(retenido_w4 * 100.0 / total_usuarios, 2) AS semana_4
FROM resumen
ORDER BY mes_cohorte;
'''

cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

In [ ]:
# ¿Qué planes existen?
print(users["tipo_plan"].value_counts())

In [ ]:
query_retencion_plan = '''
WITH actividad AS (
    SELECT
        u.id_usuario,
        u.tipo_plan,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 1  AND 7
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w1,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 8  AND 14
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w2,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 15 AND 21
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w3,
        SUM(CASE WHEN ua.dias_despues_registro BETWEEN 22 AND 28
                 AND CAST(ua.activo AS INT) = 1 THEN 1 ELSE 0 END) AS retenido_w4
    FROM users u
    LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
    GROUP BY u.id_usuario, u.tipo_plan
),
resumen AS (
    SELECT
        tipo_plan,
        COUNT(id_usuario)  AS total_usuarios,
        SUM(retenido_w1)   AS retenido_w1,
        SUM(retenido_w2)   AS retenido_w2,
        SUM(retenido_w3)   AS retenido_w3,
        SUM(retenido_w4)   AS retenido_w4
    FROM actividad
    GROUP BY tipo_plan
)
SELECT
    tipo_plan,
    total_usuarios,
    ROUND(retenido_w1 * 100.0 / total_usuarios, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / total_usuarios, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / total_usuarios, 2) AS semana_3,
    ROUND(retenido_w4 * 100.0 / total_usuarios, 2) AS semana_4
FROM resumen
ORDER BY tipo_plan;
'''

retencion_plan = pd.read_sql(query_retencion_plan, con=engine)
retencion_plan

In [ ]:
# Preparar datos para heatmap por cohorte mensual
heatmap_data = cohorte_final[
    ["mes_cohorte", "semana_1", "semana_2", "semana_3", "semana_4"]
].copy()

# Formatear las fechas para que se vean mejor
heatmap_data["mes_cohorte"] = heatmap_data["mes_cohorte"].dt.strftime("%Y-%m")
heatmap_data = heatmap_data.set_index("mes_cohorte")
heatmap_data.columns = ["Semana 1", "Semana 2", "Semana 3", "Semana 4"]

# Graficar
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Retención por cohorte — RappiPlus", fontsize=13)

# ── Heatmap 1: Por cohorte mensual ──────────────────────────
sns.heatmap(
    heatmap_data,          # el que ya creaste antes
    annot=True,
    fmt=".1f",
    cmap="YlGn",
    vmin=35, vmax=65,
    linewidths=0.5,
    linecolor="white",
    ax=axes[0],
    cbar_kws={"label": "% Retención"}
)
axes[0].set_title("Por cohorte mensual")
axes[0].set_xlabel("Semana de retención")
axes[0].set_ylabel("Cohorte (mes de registro)")
axes[0].tick_params(axis="x", rotation=0)
axes[0].tick_params(axis="y", rotation=0)

# Preparar datos para heatmap por tipo de plan
heatmap_plan = retencion_plan[
    ["semana_1", "semana_2", "semana_3", "semana_4"]
].copy()

# Usar tipo_plan como índice
heatmap_plan.index = retencion_plan["tipo_plan"]
heatmap_plan.columns = ["Semana 1", "Semana 2", "Semana 3", "Semana 4"]

# ── Heatmap 2: Por tipo de plan ─────────────────────────────
sns.heatmap(
    heatmap_plan,
    annot=True,
    fmt=".1f",
    cmap="YlGn",
    vmin=35, vmax=65,      # misma escala para comparar
    linewidths=0.5,
    linecolor="white",
    ax=axes[1],
    cbar_kws={"label": "% Retención"}
)
axes[1].set_title("Por tipo de plan")
axes[1].set_xlabel("Semana de retención")
axes[1].set_ylabel("Plan")
axes[1].tick_params(axis="x", rotation=0)
axes[1].tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

**📋 Hallazgos — Retención por cohortes**

**Configuración técnica**
- Tablas utilizadas: `users` (8.000 registros) y
  `user_activity` (32.000 registros)
- `fecha_registro` convertida con `CAST(fecha_registro AS DATE)`
  directamente en SQL
- Período de análisis: 4 semanas (días 1-28), determinado por
  el máximo disponible en `user_activity`

**Retención global por cohorte**

La retención se mantiene **notablemente estable** entre el 40-44%
durante las 4 semanas desde el registro.

**Retención segmentada por tipo de plan**

El análisis por plan revela la verdadera dinámica del negocio:

| Plan | Usuarios | % base | S1 | S2 | S3 | S4 |
|---|---|---|---|---|---|---|
| free | 5.968 | 74.6% | 35.05% | 35.82% | 35.66% | 34.89% |
| paid | 2.032 | 25.4% | 62.40% | 59.89% | 60.14% | 57.48% |

**Hallazgos Principales**

- **Brecha de retención significativa:** los usuarios `paid` retienen
~27 puntos porcentuales más que los `free` en todas las semanas,
siendo casi el doble de fieles al producto.

- **Estabilidad en ambos segmentos:** tanto `free` como `paid`
mantienen su retención prácticamente plana durante las 4 semanas,
lo que indica que el producto genera valor sostenido independientemente
del plan.

**La retención global cercana al 41% refleja un promedio ponderado entre ambos planes:**

 **💡 Recomendación**
Convertir usuarios `free` → `paid` es la palanca de retención
más poderosa disponible. Con solo el 25.4% de usuarios en plan
`paid` y una brecha de ~27 puntos de retención, existe un alto
potencial de mejora mediante estrategias de conversión como:
- Trials gratuitos del plan paid
- Descuentos por primer mes
- Comunicación del valor diferencial del plan paid

---


## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  


---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La nueva UI del checkout **no genera**
  diferencia significativa en la tasa de conversión entre el grupo
  control y el grupo tratamiento.
   - **H₁ (Hipótesis alternativa):** La nueva UI del checkout **sí genera**
  una diferencia significativa en la tasa de conversión entre el grupo
  control y el grupo tratamiento.
   
**Test estadístico:** Z-test de proporciones de dos muestras
— adecuado porque la métrica `convirtio` es binaria (0/1)
y se comparan dos grupos independientes.

**Nivel de significancia alpha:** α = 0.05
— si el p-valor < 0.05 se rechaza H₀

In [ ]:
# Cargar librerias y archivos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
experiment_checkout = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment_checkout.info()

In [ ]:
# ── Limpieza experiment_checkout
experiment_clean = experiment_checkout.copy()

# 1. Convertir timestamp a datetime
experiment_clean["timestamp"] = pd.to_datetime(
    experiment_clean["timestamp"], errors="coerce"
)
print(f"NaT tras conversión: {experiment_clean['timestamp'].isna().sum()}")

In [ ]:
# 2. Estandarizar categóricas
categoricas = ["variante", "dispositivo", "pais"]

for col in categoricas:
    experiment_clean[col] = (
        experiment_clean[col]
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    print(f"[OK] {col}: {experiment_clean[col].unique()}")

In [ ]:
# 3. Verificar convirtio (debe ser 0 o 1 únicamente)
print(f"\nValores únicos en convirtio: {experiment_clean['convirtio'].unique()}")
print(f"Distribución:\n{experiment_clean['convirtio'].value_counts()}")

In [ ]:
# 4. Duplicados
print(f"\nDuplicados: {experiment_clean.duplicated().sum()}")

In [ ]:
# 5. Resumen final
experiment_clean.info()

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import scipy.stats as stats

# ── Separar grupos
control     = experiment_clean[experiment_clean["variante"] == "control"]
tratamiento = experiment_clean[experiment_clean["variante"] == "tratamiento"]

# ── Métricas por grupo
conv_control     = control["convirtio"].sum()
conv_tratamiento = tratamiento["convirtio"].sum()
n_control        = len(control)
n_tratamiento    = len(tratamiento)

tasa_control     = conv_control / n_control
tasa_tratamiento = conv_tratamiento / n_tratamiento

print(f"Control     → n: {n_control:,} | conversiones: {conv_control} | tasa: {tasa_control:.4f} ({tasa_control*100:.2f}%)")
print(f"Tratamiento → n: {n_tratamiento:,} | conversiones: {conv_tratamiento} | tasa: {tasa_tratamiento:.4f} ({tasa_tratamiento*100:.2f}%)")
print(f"Diferencia  → {(tasa_tratamiento - tasa_control)*100:.2f} puntos porcentuales")


In [ ]:
# ── Z-test de proporciones
conteos = [conv_tratamiento, conv_control]
nobs    = [n_tratamiento, n_control]

stat, p_value = proportions_ztest(conteos, nobs)

print(f"\nZ-statistic: {stat:.4f}")
print(f"P-valor:     {p_value:.4f}")
print(f"\nAlpha:       0.05")
print(f"Conclusión:  {'✓ Rechazar H₀ — hay diferencia significativa' if p_value < 0.05 else '✗ No rechazar H₀ — no hay diferencia significativa'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("A/B Test — Impacto de nueva UI en conversión", fontsize=13)

# ── Gráfico 1: Tasa de conversión por grupo ──────────────
grupos  = ["Control", "Tratamiento"]
tasas   = [tasa_control * 100, tasa_tratamiento * 100]
colores = ["#7F77DD", "#5DCAA5"]

bars = axes[0].bar(grupos, tasas, color=colores, width=0.4)
axes[0].set_ylim(0, 25)
axes[0].set_ylabel("Tasa de conversión (%)")
axes[0].set_title("Tasa de conversión por grupo")

# Etiquetas sobre las barras
for bar, tasa in zip(bars, tasas):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{tasa:.2f}%",
        ha="center", fontsize=11, fontweight="bold"
    )

# Línea de referencia
axes[0].axhline(y=tasa_control * 100, color="gray",
                linestyle="--", alpha=0.5, label="Baseline control")
axes[0].legend(fontsize=9)

# ── Gráfico 2: Distribución convirtio por grupo ──────────
datos_plot = experiment_clean.groupby(
    ["variante", "convirtio"]
).size().reset_index(name="count")

categorias = ["control", "tratamiento"]
x = np.arange(len(categorias))
width = 0.35

no_conv = [
    datos_plot[(datos_plot["variante"]==g) &
               (datos_plot["convirtio"]==0)]["count"].values[0]
    for g in categorias
]
si_conv = [
    datos_plot[(datos_plot["variante"]==g) &
               (datos_plot["convirtio"]==1)]["count"].values[0]
    for g in categorias
]

axes[1].bar(x - width/2, no_conv, width,
            label="No convirtió (0)", color="#D85A30", alpha=0.85)
axes[1].bar(x + width/2, si_conv, width,
            label="Convirtió (1)", color="#5DCAA5", alpha=0.85)

axes[1].set_xticks(x)
axes[1].set_xticklabels(["Control", "Tratamiento"])
axes[1].set_ylabel("Usuarios")
axes[1].set_title("Distribución de conversión por grupo")
axes[1].legend(fontsize=9)

# Etiquetas
for i, (n, s) in enumerate(zip(no_conv, si_conv)):
    axes[1].text(i - width/2, n + 30, f"{n:,}",
                 ha="center", fontsize=9)
    axes[1].text(i + width/2, s + 30, f"{s:,}",
                 ha="center", fontsize=9)

# Anotación del p-valor
fig.text(0.5, -0.02,
         f"Z-statistic: 0.8133 | P-valor: 0.4161 | α: 0.05 → No significativo",
         ha="center", fontsize=10, color="gray", style="italic")

plt.tight_layout()
plt.show()

**📋 Hallazgos — Test estadístico A/B (UI Checkout)**

**Configuración del experimento**
- Dataset: 10.000 usuarios · 0 nulos · 0 duplicados
- Grupos: control (4.965) vs tratamiento (5.035)
- Métrica principal: `convirtio` (1 = compró · 0 = no compró)
- Test aplicado: Z-test de proporciones · α = 0.05

**Hipótesis**
- **H₀:** La nueva UI no genera diferencia significativa en la tasa de conversión
- **H₁:** La nueva UI sí genera diferencia significativa en la tasa de conversión

**Resultados**

| Grupo | Usuarios | Conversiones | Tasa |
|---|---|---|---|
| Control | 4.965 | 779 | 15.69% |
| Tratamiento | 5.035 | 820 | 16.29% |
| **Diferencia** | | | **+0.60 pp** |



| Z-statistic | P-valor | Alpha | Conclusión |
|---|---|---|---|
| 0.8133 | 0.4161 | 0.05 | **No rechazar H₀** |

**Conclusión**
El test A/B indica que la nueva UI del checkout no generó un impacto estadísticamente significativo en la tasa de conversión. Aunque el grupo tratamiento mostró una conversión ligeramente superior al control (+0.60 pp), el resultado no fue significativo (p = 0.4161 > 0.05), por lo que no existe evidencia suficiente para atribuir esa variación al cambio en la interfaz. Con la evidencia actual, no se recomienda implementar la nueva UI de forma generalizada.

**💡 Recomendación**
- No implementar el cambio de UI con la evidencia actual
- Rediseñar el experimento con mayor diferenciación en la UI
- Segmentar por dispositivo o país para detectar
  efectos en subgrupos específicos

---



## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc



**Conclusión final**

Se resolvieron las principales preguntas del negocio: los datos fueron depurados y validados, RappiPlus mostró rentabilidad positiva, el mayor abandono se detectó antes del ingreso de datos de pago, la retención fue más fuerte en usuarios paid y el experimento A/B no mostró evidencia suficiente de impacto significativo. Como entregable final, se construyó un dashboard que resume estos resultados de forma visual y ejecutiva.

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

https://public.tableau.com/views/Proyecto_Final_Dayana_Rodriguez/Overview?:language=es-ES&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link